In [1]:
# 1. SETUP & IMPORTS
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib  # For saving models
import os

# Scikit-Learn Imports
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# --- CONFIGURATION ---
BASE_PATH = os.path.dirname(os.getcwd())
INPUT_FEATURES = f'{BASE_PATH}/datasets/X_features.csv'
INPUT_TARGETS = f'{BASE_PATH}/datasets/y_targets.csv'
MODEL_DIR = f'{BASE_PATH}/models'

# Create model directory if it doesn't exist
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

print(f"Base Path: {BASE_PATH}")
print(f"Model Directory: {MODEL_DIR}")

Base Path: /Users/nirvik/final_year_project/orange_freshness_detection
Model Directory: /Users/nirvik/final_year_project/orange_freshness_detection/models


In [3]:
# 2. LOAD & PREPARE DATA WITH PROPER TRAIN/TEST SPLIT
# ==========================================
print("="*60)
print("LOADING DATA")
print("="*60)

X = pd.read_csv(INPUT_FEATURES)
y = pd.read_csv(INPUT_TARGETS)

print(f"Total dataset shape: {X.shape}")
print(f"All batches in data: {list(y['batch'].unique())[:10]}")  # Show sample to avoid sorting mixed types
print(f"Day range: {y['day'].min()}-{y['day'].max()}")

# Handle mixed batch types (integers 1-7 and 'Synthetic')
y['batch_str'] = y['batch'].astype(str)

# Train: Real batches 1-5 + ALL synthetic data (they don't have batch identity)
# Test: Real batches 6-7 only (completely unseen real data)
train_mask = (y['batch_str'].isin(['1', '2', '3', '4', '5', 'Synthetic']))
test_mask = (y['batch_str'].isin(['6', '7']))

X_train = X[train_mask].copy()
X_test = X[test_mask].copy()

y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

y_quant_train = y_train['day']        # Regression: storage days
y_quant_test = y_test['day']

print("\n" + "="*60)
print("DATA SPLIT SUMMARY (DAY MODEL)")
print("="*60)
print(f"Training set: {X_train.shape[0]} samples")
print(f"  - Synthetic samples: {(y_train['batch_str'] == 'Synthetic').sum()}")
print(f"  - Real batches 1-5: {(y_train['batch_str'].isin(['1','2','3','4','5'])).sum()}")
print(f"Testing set: {X_test.shape[0]} samples from real batches 6-7")
print(f"\nTrain day range: {y_quant_train.min():.2f}-{y_quant_train.max():.2f}")
print(f"Test day range: {y_quant_test.min():.2f}-{y_quant_test.max():.2f}")

# Scale the features (Important for tree + distance models downstream)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Prepare folic acid regression dataset (uses only labeled real samples)
conc_available = 'true_conc_uM' in y.columns
if conc_available:
    conc_mask = y['true_conc_uM'].notna()
    conc_batches = y.loc[conc_mask, 'batch'].astype(int)
    conc_train_mask = conc_mask & (conc_batches <= 4)  # hold out batch 5 for test
    conc_test_mask = conc_mask & (conc_batches == 5)

    X_conc_train = X[conc_train_mask].copy()
    X_conc_test = X[conc_test_mask].copy()
    y_conc_train = y.loc[conc_train_mask, 'true_conc_uM'].copy()
    y_conc_test = y.loc[conc_test_mask, 'true_conc_uM'].copy()

    X_conc_train_scaled = scaler.transform(X_conc_train)
    X_conc_test_scaled = scaler.transform(X_conc_test)

    print("\n" + "="*60)
    print("DATA SPLIT SUMMARY (FOLIC ACID MODEL)")
    print("="*60)
    print(f"Training set (batches ≤4): {X_conc_train.shape[0]} samples")
    print(f"Testing set (batch 5 hold-out): {X_conc_test.shape[0]} samples")
    print(f"Train conc range: {y_conc_train.min():.2f}-{y_conc_train.max():.2f} µM")
    print(f"Test conc range: {y_conc_test.min():.2f}-{y_conc_test.max():.2f} µM")
else:
    print("\nWarning: 'true_conc_uM' not found in targets; folic acid model will be skipped.")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print("✓ Features scaled using StandardScaler")
print("✓ Ready for model training")

LOADING DATA
Total dataset shape: (2050, 11)
All batches in data: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Day range: 0.0-14.0

DATA SPLIT SUMMARY (DAY MODEL)
Training set: 1035 samples
  - Synthetic samples: 0
  - Real batches 1-5: 1035
Testing set: 415 samples from real batches 6-7

Train day range: 0.00-13.98
Test day range: 0.00-14.00

DATA SPLIT SUMMARY (FOLIC ACID MODEL)
Training set (batches ≤4): 28 samples
Testing set (batch 5 hold-out): 7 samples
Train conc range: 21.04-200.00 µM
Test conc range: 29.59-175.00 µM

PREPROCESSING COMPLETE
✓ Features scaled using StandardScaler
✓ Ready for model training


In [4]:
# 3. FEATURE SELECTION (RFE)
# ==========================================
print("\n" + "="*60)
print("FEATURE SELECTION: Recursive Feature Elimination (Day target)")
print("="*60)

selector = RFE(estimator=RandomForestRegressor(n_estimators=150, random_state=42), 
               n_features_to_select=5, step=1)
selector = selector.fit(X_train_scaled, y_quant_train)

# Get the selected feature names
selected_indices = selector.get_support(indices=True)
selected_features = X.columns[selected_indices]

print(f"Top 5 Selected Features: {list(selected_features)}")

# Apply selection
X_train_opt = X_train_scaled[:, selected_indices]
X_test_opt = X_test_scaled[:, selected_indices]

print(f"Optimized training shape: {X_train_opt.shape}")
print(f"Optimized testing shape: {X_test_opt.shape}")


FEATURE SELECTION: Recursive Feature Elimination (Day target)
Top 5 Selected Features: ['Mean', 'Energy', 'DCT_1', 'DCT_2', 'DCT_3']
Optimized training shape: (1035, 5)
Optimized testing shape: (415, 5)


In [ ]:
# 4.  FOLIC ACID REGRESSION (true_conc_uM)
# ==========================================
print("\n" + "="*60)
print("TRAINING FOLIC ACID REGRESSION MODEL")
print("="*60)

conc_reg = None
if conc_available and X_conc_train.shape[0] > 0 and X_conc_test.shape[0] > 0:
    X_conc_train_opt = X_conc_train_scaled[:, selected_indices]
    X_conc_test_opt = X_conc_test_scaled[:, selected_indices]

    conc_reg = RandomForestRegressor(n_estimators=300, random_state=42)
    conc_reg.fit(X_conc_train_opt, y_conc_train)

    conc_pred = conc_reg.predict(X_conc_test_opt)
    conc_rmse = np.sqrt(mean_squared_error(y_conc_test, conc_pred))
    conc_mae = np.mean(np.abs(y_conc_test - conc_pred))
    conc_r2 = r2_score(y_conc_test, conc_pred)

    print(f"✓ Model trained on batches 1-4; tested on hold-out batch 5")
    print(f"\nFolic Acid Prediction Performance (µM):")
    print(f"  RMSE: ±{conc_rmse:.2f} µM")
    print(f"  MAE: ±{conc_mae:.2f} µM")
    print(f"  R² Score: {conc_r2:.4f}")
else:
    print("No folic acid labels with both train and test splits; skipping folic model.")


TRAINING FOLIC ACID REGRESSION MODEL
✓ Model trained on batches 1-4; tested on hold-out batch 5

Folic Acid Prediction Performance (µM):
  RMSE: ±0.79 µM
  MAE: ±0.73 µM
  R² Score: 0.9997


In [ ]:
# 5.  REGRESSION (Storage Day Prediction)
# ==========================================
print("\n" + "="*60)
print("TRAINING REGRESSION MODEL (Storage Day Prediction)")
print("="*60)

reg = RandomForestRegressor(n_estimators=200, random_state=42)
reg.fit(X_train_opt, y_quant_train)

# Evaluate
y_quant_pred = reg.predict(X_test_opt)
rmse = np.sqrt(mean_squared_error(y_quant_test, y_quant_pred))
mae = np.mean(np.abs(y_quant_test - y_quant_pred))
r2 = r2_score(y_quant_test, y_quant_pred)

print(f"✓ Model trained on batches 1-5 + synthetic; tested on completely unseen batches 6-7")
print(f"\nDay Prediction Performance:")
print(f"  RMSE: ±{rmse:.2f} days")
print(f"  MAE: ±{mae:.2f} days")
print(f"  R² Score: {r2:.4f}")
print(f"\nInterpretation:")
print(f"  - On average, predictions are off by {mae:.1f} days")
print(f"  - Day range in test set: {y_quant_test.min()}-{y_quant_test.max()}")
if r2 > 0.7:
    print(f"  - ✓ Excellent predictive performance!")
elif r2 > 0.5:
    print(f"  - Good predictive performance")
elif r2 > 0.3:
    print(f"  - Moderate predictive performance")
else:
    print(f"  - Limited predictive performance (may need more training data)")


TRAINING REGRESSION MODEL (Storage Day Prediction)
✓ Model trained on batches 1-5 + synthetic; tested on completely unseen batches 6-7

Day Prediction Performance:
  RMSE: ±1.65 days
  MAE: ±1.14 days
  R² Score: 0.8372

Interpretation:
  - On average, predictions are off by 1.1 days
  - Day range in test set: 0.0-14.0
  - ✓ Excellent predictive performance!


In [7]:
# 6. SAVE MODELS
# ==========================================
print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

if conc_reg is not None:
    joblib.dump(conc_reg, f'{MODEL_DIR}/model_folic_rf.pkl')
    print("✓ Folic acid regression model saved")
else:
    print("⚠️ Folic acid model not saved (not trained)")

joblib.dump(reg, f'{MODEL_DIR}/model_day_rf.pkl')
joblib.dump(scaler, f'{MODEL_DIR}/scaler.pkl')
joblib.dump(selected_indices, f'{MODEL_DIR}/selected_features.pkl')

print("✓ Day regression model saved")
print("✓ Scaler saved")
print("✓ Selected features saved")
print(f"\nAll models saved to: {MODEL_DIR}/")
print("\n" + "="*60)
print("MODELING COMPLETE!")
print("="*60)
print("\nKey Outputs:")
print("1. Day prediction model (unseen batches 6-7)")
print("2. Folic acid model (trained on batches 1-4, tested on batch 5 if labels available)")


SAVING MODELS
✓ Folic acid regression model saved
✓ Day regression model saved
✓ Scaler saved
✓ Selected features saved

All models saved to: /Users/nirvik/final_year_project/orange_freshness_detection/models/

MODELING COMPLETE!

Key Outputs:
1. Day prediction model (unseen batches 6-7)
2. Folic acid model (trained on batches 1-4, tested on batch 5 if labels available)
